# Fair comparison: Table + Graph vs Edge-aware Tabular + GNN

This notebook trains two **controlled, reproducible implementations** on the same data:
1. **TableGraph baseline**: tabular branch + GCN molecular graph branch (does not use bond attributes).
2. **Edge-aware GNN**: same tabular branch + GINE layers that use bond/edge attributes.

It evaluates each model on (A) one shared random row split and (B) one shared **canonical-SMILES-grouped split**. Validation RMSE selects the best epoch; test data is used only for final evaluation.

**Important:** this is a fair re-training comparison, not a way to load the old `.pt` checkpoint from the prior custom Edge-aware notebook. Exact checkpoint comparison requires its matching model class and preprocessing code. The repository's `tableGraphTrainGPU.py` uses a boosting ensemble; this notebook's TableGraph is a simpler GCN baseline, so label it as a *GCN TableGraph reimplementation*, not the exact published TableGraph, unless you port the original ensemble into this split framework.

## 1. Install dependencies and upload the repository ZIP
Run this cell, then upload `MolGBN-OPR-master.zip` when prompted. The notebook will use `data/processed/MemTrOC-Dataset.csv` from the repository if present.

In [ ]:
!pip -q install rdkit torch-geometric scikit-learn pandas numpy matplotlib
from google.colab import files
uploaded = files.upload()

In [ ]:
import os, zipfile, glob, json, random, copy, warnings
import numpy as np
import pandas as pd
from rdkit import Chem
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, GINEConv, global_mean_pool

SEED = 41
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

zip_files = [f for f in uploaded if f.lower().endswith('.zip')]
if zip_files:
    with zipfile.ZipFile(zip_files[0]) as zf:
        zf.extractall('/content/project')
csv_candidates = glob.glob('/content/project/**/MemTrOC-Dataset.csv', recursive=True)
if not csv_candidates:
    csv_candidates = glob.glob('/content/project/**/*.csv', recursive=True)
print('CSV candidates:', csv_candidates[:10])
assert csv_candidates, 'No CSV found. Upload the project ZIP or a dataset CSV.'
DATA_PATH = csv_candidates[0]
df = pd.read_csv(DATA_PATH)
print('Using:', DATA_PATH, '| shape:', df.shape)
display(df.head())
print('Columns:', list(enumerate(df.columns)))

## 2. Configure the dataset columns

The supplied repository's training script assumes SMILES at column index 3, tabular features at indices 4–22, and target at index 23. The defaults below follow that convention. If your CSV differs, change these values after inspecting the printed columns.

The feature set is numeric tabular columns only. SMILES and target are excluded. Rows with missing target, missing SMILES, invalid SMILES, or missing numeric features are dropped before splitting.

In [ ]:
# Repository defaults; adjust if your uploaded CSV uses a different schema.
SMILES_COL = df.columns[3] if df.shape[1] > 3 else None
TARGET_COL = df.columns[23] if df.shape[1] > 23 else df.columns[-1]
FEATURE_COLS = list(df.columns[4:23]) if df.shape[1] > 23 else [
    c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET_COL
]
print('SMILES_COL:', SMILES_COL)
print('TARGET_COL:', TARGET_COL)
print('FEATURE_COLS:', FEATURE_COLS)

assert SMILES_COL in df.columns and TARGET_COL in df.columns
assert FEATURE_COLS, 'No numeric feature columns found; set FEATURE_COLS manually.'

work = df[[SMILES_COL, TARGET_COL] + FEATURE_COLS].copy()
work[SMILES_COL] = work[SMILES_COL].astype(str).str.strip()
work[TARGET_COL] = pd.to_numeric(work[TARGET_COL], errors='coerce')
for c in FEATURE_COLS:
    work[c] = pd.to_numeric(work[c], errors='coerce')

def canonicalize(s):
    mol = Chem.MolFromSmiles(s)
    return Chem.MolToSmiles(mol, canonical=True) if mol is not None else None

work['canonical_smiles'] = work[SMILES_COL].map(canonicalize)
work = work.dropna(subset=['canonical_smiles', TARGET_COL] + FEATURE_COLS).reset_index(drop=True)
print('Rows after cleaning:', len(work))
print('Unique canonical SMILES:', work['canonical_smiles'].nunique())
print('Feature count:', len(FEATURE_COLS))

## 3. Make two split protocols

Both models will use the **same exact indices** within each protocol:
- Random row split: 70% train, 15% validation, 15% test, seeded.
- Grouped split: canonical SMILES groups are assigned so the same canonical molecule cannot occur in more than one split.

Grouped splitting can be harder and can produce different distributions. Compare the models within each split protocol; don't compare a random-split score directly with a grouped-split score as if they were equivalent.

In [ ]:
def random_split_indices(n, seed=SEED):
    idx = np.arange(n)
    trainval, test = train_test_split(idx, test_size=0.15, random_state=seed)
    train, val = train_test_split(trainval, test_size=0.15/0.85, random_state=seed)
    return {'train': np.array(train), 'val': np.array(val), 'test': np.array(test)}

def grouped_split_indices(groups, seed=SEED):
    idx = np.arange(len(groups))
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=seed)
    trainval_pos, test_pos = next(gss1.split(idx, groups=groups))
    trainval_idx, test_idx = idx[trainval_pos], idx[test_pos]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=seed+1)
    tr_pos, va_pos = next(gss2.split(trainval_idx, groups=np.asarray(groups)[trainval_idx]))
    return {'train': trainval_idx[tr_pos], 'val': trainval_idx[va_pos], 'test': test_idx}

splits = {
    'random_row': random_split_indices(len(work)),
    'smiles_grouped': grouped_split_indices(work['canonical_smiles'].to_numpy())
}
for name, sp in splits.items():
    print('\n', name, {k: len(v) for k,v in sp.items()})
    if name == 'smiles_grouped':
        sets = {k:set(work.iloc[v]['canonical_smiles']) for k,v in sp.items()}
        print('Canonical SMILES overlap train/val:', len(sets['train'] & sets['val']))
        print('Canonical SMILES overlap train/test:', len(sets['train'] & sets['test']))
        print('Canonical SMILES overlap val/test:', len(sets['val'] & sets['test']))

## 4. Molecular graph features

Atom features and bond features are constructed consistently for both models. The baseline GCN ignores bond attributes; the edge-aware GINE uses them. Unknown categorical values are safely mapped to zero.

In [ ]:
ATOM_SYMBOLS = ['C','N','O','S','F','P','Cl','Br','I','B','Si','Se','Na','K','Li','Ca','Mg','Fe','Zn','Cu','Other']
HYBRIDIZATIONS = [Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
                  Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
                  Chem.rdchem.HybridizationType.SP3D2]
BOND_TYPES = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
              Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]

def one_hot_unknown(value, choices):
    return [float(value == c) for c in choices] + [float(value not in choices)]

def atom_features(atom):
    return (
        one_hot_unknown(atom.GetSymbol(), ATOM_SYMBOLS) +
        one_hot_unknown(atom.GetHybridization(), HYBRIDIZATIONS) +
        [float(atom.GetIsAromatic()), float(atom.GetIsInRing()),
         float(atom.GetFormalCharge()), float(atom.GetTotalDegree()),
         float(atom.GetTotalNumHs()), float(atom.GetMass())/200.0]
    )

def bond_features(bond):
    return (
        one_hot_unknown(bond.GetBondType(), BOND_TYPES) +
        [float(bond.GetIsConjugated()), float(bond.IsInRing()),
         float(bond.GetStereo() != Chem.rdchem.BondStereo.STEREONONE)]
    )

ATOM_DIM = len(atom_features(Chem.MolFromSmiles('C').GetAtomWithIdx(0)))
EDGE_DIM = len(bond_features(Chem.MolFromSmiles('CC').GetBondWithIdx(0)))
print('Atom feature dim:', ATOM_DIM, '| Edge feature dim:', EDGE_DIM)

_graph_cache = {}
def smiles_to_graph(smiles):
    if smiles in _graph_cache:
        return _graph_cache[smiles]
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError('Invalid SMILES: ' + str(smiles))
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float32)
    edges, edge_attrs = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        ef = bond_features(b)
        edges.extend([[i,j],[j,i]])
        edge_attrs.extend([ef,ef])
    if edges:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attrs, dtype=torch.float32)
    else:
        edge_index = torch.empty((2,0), dtype=torch.long)
        edge_attr = torch.empty((0,EDGE_DIM), dtype=torch.float32)
    g = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    _graph_cache[smiles] = g
    return g

## 5. Dataset, models, training, and validation-RMSE checkpointing

Each model receives identical rows, scaled tabular features, graph inputs, and training settings. Scaling is fit on the training rows only. Early stopping and best checkpoint selection use validation RMSE, never test RMSE.

In [ ]:
class MolTabDataset(Dataset):
    def __init__(self, frame, indices, scaler):
        self.frame = frame.iloc[indices].reset_index(drop=True)
        self.x_table = scaler.transform(self.frame[FEATURE_COLS].to_numpy(dtype=np.float32)).astype(np.float32)
        self.y = self.frame[TARGET_COL].to_numpy(dtype=np.float32)
        self.smiles = self.frame['canonical_smiles'].tolist()
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        g = smiles_to_graph(self.smiles[i])
        return torch.tensor(self.x_table[i]), g, torch.tensor([self.y[i]], dtype=torch.float32), self.smiles[i]

def collate_fn(items):
    xt, graphs, ys, smiles = zip(*items)
    return torch.stack(xt), Batch.from_data_list(list(graphs)), torch.stack(ys), list(smiles)

class TableBranch(nn.Module):
    def __init__(self, n_features, hidden, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, hidden), nn.ReLU(),
                                 nn.Dropout(dropout), nn.Linear(hidden, hidden), nn.ReLU())
    def forward(self, x): return self.net(x)

class TableGraphGCN(nn.Module):
    """GCN molecular branch + tabular branch; edge attributes are not consumed."""
    def __init__(self, n_features, hidden=128, layers=2, dropout=0.1):
        super().__init__()
        self.table = TableBranch(n_features, hidden, dropout)
        self.atom_in = nn.Linear(ATOM_DIM, hidden)
        self.convs = nn.ModuleList([GCNConv(hidden, hidden) for _ in range(layers)])
        self.dropout = dropout
        self.head = nn.Sequential(nn.Linear(2*hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, xt, graph):
        h = F.relu(self.atom_in(graph.x))
        for conv in self.convs:
            h = F.relu(conv(h, graph.edge_index))
            h = F.dropout(h, p=self.dropout, training=self.training)
        pooled = global_mean_pool(h, graph.batch)
        return self.head(torch.cat([self.table(xt), pooled], dim=1))

class EdgeAwareTabularGNN(nn.Module):
    """GINE molecular branch consumes bond/edge features + tabular branch."""
    def __init__(self, n_features, hidden=128, layers=2, dropout=0.1):
        super().__init__()
        self.table = TableBranch(n_features, hidden, dropout)
        self.atom_in = nn.Linear(ATOM_DIM, hidden)
        self.edge_in = nn.Linear(EDGE_DIM, hidden)
        self.convs = nn.ModuleList([
            GINEConv(nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden)),
                     edge_dim=hidden) for _ in range(layers)
        ])
        self.dropout = dropout
        self.head = nn.Sequential(nn.Linear(2*hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, xt, graph):
        h = F.relu(self.atom_in(graph.x))
        e = self.edge_in(graph.edge_attr) if graph.edge_attr.numel() else graph.edge_attr.new_empty((0, h.shape[1]))
        for conv in self.convs:
            h = F.relu(conv(h, graph.edge_index, e))
            h = F.dropout(h, p=self.dropout, training=self.training)
        pooled = global_mean_pool(h, graph.batch)
        return self.head(torch.cat([self.table(xt), pooled], dim=1))

def metrics(y, pred):
    return {'R2': float(r2_score(y, pred)), 'RMSE': float(np.sqrt(mean_squared_error(y, pred))),
            'MAE': float(mean_absolute_error(y, pred))}

def evaluate(model, loader):
    model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for xt, graph, y, _ in loader:
            xt, graph = xt.to(DEVICE), graph.to(DEVICE)
            p = model(xt, graph)
            ys.extend(y.numpy().reshape(-1)); ps.extend(p.cpu().numpy().reshape(-1))
    return metrics(np.asarray(ys), np.asarray(ps)), np.asarray(ys), np.asarray(ps)

def train_one(model_class, frame, split, model_name, max_epochs=250, patience=30,
              batch_size=64, lr=1e-3, hidden=128, layers=2, dropout=0.1):
    # Fit preprocessing only on training rows
    scaler = StandardScaler()
    scaler.fit(frame.iloc[split['train']][FEATURE_COLS].to_numpy(dtype=np.float32))
    ds = {k: MolTabDataset(frame, split[k], scaler) for k in ['train','val','test']}
    loaders = {
        'train': DataLoader(ds['train'], batch_size=batch_size, shuffle=True, collate_fn=collate_fn),
        'val': DataLoader(ds['val'], batch_size=batch_size, shuffle=False, collate_fn=collate_fn),
        'test': DataLoader(ds['test'], batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    }
    model = model_class(len(FEATURE_COLS), hidden=hidden, layers=layers, dropout=dropout).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=8)
    best_rmse = float('inf'); best_state = None; best_epoch = 0; wait = 0
    history=[]
    for epoch in range(1, max_epochs+1):
        model.train(); train_losses=[]
        for xt, graph, y, _ in loaders['train']:
            xt, graph, y = xt.to(DEVICE), graph.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            pred = model(xt, graph)
            loss = F.mse_loss(pred, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            train_losses.append(float(loss.item()))
        val_metrics, _, _ = evaluate(model, loaders['val'])
        scheduler.step(val_metrics['RMSE'])
        history.append({'epoch':epoch, 'train_mse':float(np.mean(train_losses)), 'val_rmse':val_metrics['RMSE']})
        if val_metrics['RMSE'] < best_rmse:
            best_rmse = val_metrics['RMSE']; best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict()); wait = 0
        else:
            wait += 1
        if epoch % 10 == 0:
            print(f'{model_name} epoch {epoch}: val RMSE={val_metrics["RMSE"]:.4f}; best={best_rmse:.4f} @ {best_epoch}')
        if wait >= patience: break
    model.load_state_dict(best_state)
    val_m, yv, pv = evaluate(model, loaders['val'])
    test_m, yt, pt = evaluate(model, loaders['test'])
    return {'model':model, 'scaler':scaler, 'history':pd.DataFrame(history),
            'best_epoch':best_epoch, 'val':val_m, 'test':test_m,
            'test_y':yt, 'test_pred':pt, 'val_y':yv, 'val_pred':pv}

## 6. Run both models on both splits

This can take a while on CPU; use a Colab GPU runtime if available. To keep the comparison fair, both models use the same epoch limit, patience, hidden size, layer count, batch size, and learning rate. For publication-quality results, repeat with several seeds and report mean ± standard deviation.

In [ ]:
MAX_EPOCHS = 250
PATIENCE = 30
BATCH_SIZE = 64
HIDDEN = 128
LAYERS = 2
DROPOUT = 0.10
LR = 1e-3

model_specs = {
    'TableGraph_GCN': TableGraphGCN,
    'EdgeAware_GINE': EdgeAwareTabularGNN
}
all_runs = {}
summary_rows = []
prediction_frames = []

for split_name, split in splits.items():
    for model_name, model_class in model_specs.items():
        print('\n' + '='*70 + f'\n{split_name} | {model_name}\n' + '='*70)
        result = train_one(model_class, work, split, model_name,
                           max_epochs=MAX_EPOCHS, patience=PATIENCE,
                           batch_size=BATCH_SIZE, lr=LR, hidden=HIDDEN,
                           layers=LAYERS, dropout=DROPOUT)
        all_runs[(split_name, model_name)] = result
        summary_rows.append({'split':split_name, 'model':model_name, 'best_epoch':result['best_epoch'],
                             **{'val_'+k:v for k,v in result['val'].items()},
                             **{'test_'+k:v for k,v in result['test'].items()}})
        test_idx = split['test']
        pred_df = work.iloc[test_idx][[SMILES_COL, 'canonical_smiles', TARGET_COL]].copy()
        pred_df['split'] = split_name
        pred_df['model'] = model_name
        pred_df['actual'] = result['test_y']
        pred_df['prediction'] = result['test_pred']
        prediction_frames.append(pred_df)

summary = pd.DataFrame(summary_rows).sort_values(['split','test_RMSE'])
display(summary)

## 7. Inspect results and split leakage checks

In [ ]:
display(summary.pivot(index='model', columns='split', values=['val_RMSE','test_RMSE','test_R2','test_MAE']))
for split_name, split in splits.items():
    print('\n', split_name)
    if split_name == 'smiles_grouped':
        gsets = {k:set(work.iloc[v]['canonical_smiles']) for k,v in split.items()}
        assert not (gsets['train'] & gsets['val'])
        assert not (gsets['train'] & gsets['test'])
        assert not (gsets['val'] & gsets['test'])
        print('PASS: no canonical-SMILES overlap across train/validation/test.')
    else:
        # Random row split may contain duplicate molecules across subsets; quantify it.
        gsets = {k:set(work.iloc[v]['canonical_smiles']) for k,v in split.items()}
        print('Unique molecule overlap train/test:', len(gsets['train'] & gsets['test']))

## 8. Save models, scalers, metrics, predictions, and split indices

In [ ]:
import joblib
os.makedirs('/content/comparison_outputs', exist_ok=True)
summary.to_csv('/content/comparison_outputs/metrics_summary.csv', index=False)
pd.concat(prediction_frames, ignore_index=True).to_csv('/content/comparison_outputs/test_predictions.csv', index=False)

for (split_name, model_name), result in all_runs.items():
    prefix = f'{split_name}__{model_name}'
    torch.save({
        'model_state_dict': result['model'].state_dict(),
        'model_class': model_name,
        'feature_cols': FEATURE_COLS,
        'smiles_col': SMILES_COL,
        'target_col': TARGET_COL,
        'atom_dim': ATOM_DIM,
        'edge_dim': EDGE_DIM,
        'hidden': HIDDEN, 'layers': LAYERS, 'dropout': DROPOUT,
        'best_epoch': result['best_epoch'],
        'validation_metrics': result['val'],
        'test_metrics': result['test'],
    }, f'/content/comparison_outputs/{prefix}.pt')
    joblib.dump(result['scaler'], f'/content/comparison_outputs/{prefix}_scaler.joblib')
    result['history'].to_csv(f'/content/comparison_outputs/{prefix}_history.csv', index=False)

for split_name, split in splits.items():
    pd.DataFrame({k:pd.Series(v) for k,v in split.items()}).to_csv(
        f'/content/comparison_outputs/{split_name}_split_indices.csv', index=False)

with open('/content/comparison_outputs/run_config.json','w') as f:
    json.dump({'seed':SEED, 'max_epochs':MAX_EPOCHS, 'patience':PATIENCE,
               'batch_size':BATCH_SIZE, 'hidden':HIDDEN, 'layers':LAYERS,
               'dropout':DROPOUT, 'lr':LR, 'features':FEATURE_COLS,
               'smiles_col':SMILES_COL, 'target_col':TARGET_COL}, f, indent=2)

import shutil
shutil.make_archive('/content/edge_aware_vs_tablegraph_comparison', 'zip', '/content/comparison_outputs')
from google.colab import files
files.download('/content/edge_aware_vs_tablegraph_comparison.zip')

## Interpretation notes

- Prefer the **validation RMSE** for model/epoch selection. Use test metrics only as the final estimate.
- A lower random-split RMSE may reflect repeated molecules in different subsets; grouped-SMILES performance is the stronger check for unseen-molecule generalization.
- This notebook uses canonical SMILES as the grouping key. If your rows represent salts, tautomers, stereoisomers, or multiple experiments per compound and you need those grouped together, define a stricter compound ID / standardization policy.
- For a fair benchmark against the exact published TableGraph result (R² = 0.9014, RMSE = 9.1118), port the original `DynamicNetForMLPGNN` ensemble from `models/ensemblemodel.py` and `models/weaklearner.py` into the same split/evaluation functions. Do not compare scores from different splits.